<a href="https://colab.research.google.com/github/GimenesPaula/GimenesPaula/blob/main/Lan%C3%A7amentos_Vendas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bibliotecas Phyton

In [1]:
!pip install requests

In [2]:
pip install unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 12.7 MB/s eta 0:00:00


In [326]:
pip install fuzzywuzzy

In [327]:
#Carrega Bibliotecas
import pandas as pd
import numpy as np
import re
from functools import lru_cache
from unidecode import unidecode
import requests
from collections import defaultdict
from fuzzywuzzy import fuzz

/usr/local/lib/python3.12/dist-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


# Fazer Upload planilhas
1.   Lançamentos
2.   Inci
3.   Vendas

In [4]:
#Lançamentos
from google.colab import files
uploaded = files.upload()

Saving Brasil, 30-09-25.xlsx to Brasil, 30-09-25.xlsx


In [5]:
filename = next(iter(uploaded))

In [6]:
# INCI name produtos
from google.colab import files
inci = files.upload()

Saving inci_16_08.xlsx to inci_16_08.xlsx


In [7]:
filename2 = next(iter(inci))

In [8]:
#Vendas Distribuidores
from google.colab import files
dist = files.upload()

Saving sales (1).xlsx to sales (1).xlsx


In [9]:
filename4 = next(iter(dist))

In [336]:
data = '/content/'+filename4
df_sale = pd.read_excel(data)
pd.options.display.max_columns=None
df_sale.nunique()

,0
Distributor (subsidiary),4
WACKER Material No.,63
WACKER Material Name,92
Customer Name,649
Region (State/Province) ship-to (of customer),22
Month of Invoice,9
Quantity actual period,148


# Análise Ferramenta de Vendas

In [283]:
PALAVRAS_IRRELEVANTES = {
    'industria', 'comercio', 'cosmetico', 'cosmeticos', 'tecnologia', 'ltda', 'me', 'eireli', 'sa', 'cia', 'comercial',
    'produtos', 'servicos', 'serviço', 'do', 'da', 'de', 'dos', 'das', 'the', 'group', 'grupo',
    'inc', 'corp', 'corporation', 'associados', 'associado', 'associacao', 'associação', 'holding', 'importadora',
    'exportadora', 'importacao', 'importação', 'exportacao', 'exportação', 'distribuidora', 'distribuidor', 'fabricacao',
    'fabricante', 'comerciante', 'comercio', 'comércio', 'comercial', 'empresa', 'sociedade', 'unipessoal'
}

def limpar(texto):
    if pd.isnull(texto):
        return []
    texto = unidecode(str(texto)).lower()
    texto = re.sub(r'\d+', '', texto)
    texto = re.sub(r'[.,/\\-]', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return [p for p in texto.split(' ') if p]

def extrair_chave(palavras):
    relevantes = [p for p in palavras if p not in PALAVRAS_IRRELEVANTES]
    if not relevantes:
        relevantes = palavras
    if not relevantes:
        return ''
    for p in relevantes:
        if len(p) > 1:
            return p
    return ' '.join(relevantes[:3])


def gerar_chaves(df, coluna1, coluna2=None, coluna3=None):
    palavras1 = df[coluna1].apply(limpar)
    palavras2 = df[coluna2].apply(limpar) if coluna2 else pd.Series([[]] * len(df))
    palavras3 = df[coluna3].apply(limpar) if coluna3 else pd.Series([[]] * len(df))

    def chave_final(idx):
        for lista in [palavras1.iloc[idx], palavras2.iloc[idx], palavras3.iloc[idx]]:
            chave = extrair_chave(lista)
            if chave:
                return chave
        return f"sem_chave_{idx}"  # fallback seguro

    return pd.Series([chave_final(i) for i in range(len(df))], index=df.index)


In [319]:
def formatar_material(coluna):
    def extrair_codigo(texto):
        #Tenta extrair o padrão: 2+ letras + 1+ números
        match = re.search(r'\b([A-Za-z]{2,}\s*\d{1,})\b', texto)
        if match:
            return match.group(1).upper().strip()
    return coluna.apply(extrair_codigo)

In [320]:
# Filtra apenas linhas que contenham 'BELSIL' na descrição (case-insensitive)
df_sale = df_sale[df_sale['WACKER Material Name'].str.contains('BELSIL', case=False, na=False)]

# Limpa e padroniza a coluna 'Material'
df_sale['WACKER Material Name'] = formatar_material(df_sale['WACKER Material Name'].astype(str))

# Padroniza fabricantes e distribuidores
df_sale['KeyManuf'] = gerar_chaves(df_sale, 'Customer Name')
df_sale['Estado'] = df_sale['Region (State/Province) ship-to (of customer)']


# Gera os resumos por ano
df_final = df_sale.groupby(['KeyManuf', 'Estado']).agg(
               Distributor_Set = ('Distributor (subsidiary)', lambda x: sorted(set(x))),
               Sales_Qnty_2025 = ('WACKER Material Name', lambda x: len(set(i for i in x if i))),
               Sales_Kg_2025 = ('Quantity actual period', 'sum'),
               Sales_2025 = ('WACKER Material Name', to_set),
               )

# Salva o resultado em Excel
df_final.to_excel('Resumo_vendas.xlsx')
df_final.head()

,,Distributor_Set,Sales_Qnty_2025,Sales_Kg_2025,Sales_2025
KeyManuf,Estado,,,,
&co,SP,[FOCUS QUIMICA],3,398.0,"[DM 5, EG 6000, TMS 803]"
a&s,SP,[FOCUS QUIMICA],4,3814.0,"[DM 0, DM 6010, OW 2100, TMS 803]"
abalo,SP,[FOCUS QUIMICA],6,60.0,"[ADM 9000, DM 350, DM 6010, GB 3024, GB 3025, ..."
abboccato,SC,[FOCUS QUIMICA],1,5.0,[DM 6010]
abs,BA,[MORAIS DE CASTRO],1,38.0,[GB 150]


# Análise Relatórios Lançamentos

## Descritivo Lançamentos





In [286]:
#Carrega o banco de dados como tabela
data = '/content/'+filename
df_launches = pd.read_excel(data)
pd.options.display.max_columns = None
df_launches.nunique()

,0
Número do Produto,6220
Data de Publicação,430
Produto,3741
Marca,3495
Empresa,908
Categoria,4
Sub-Categoria,39
Descrição do Produto,6219
Preço por 100g/ml,3218
Preço em moeda local,1333


In [287]:
#Edita Coluna Ano
df_launches['Ano'] = pd.to_datetime(df_launches['Data de Publicação']).dt.year

##Upload INCI

In [288]:
data = '/content/'+filename2
df_inci = pd.read_excel(data)
pd.options.display.max_columns=None
df_inci.nunique()

,0
Produto,51
Ingrediente,50
Prioridade,3


## Função Analisa Ingredientes

In [289]:
#this checks if any combination of INCI as present in Ingredient
def verifica_ingrediente(formula,produto,material):
  quantidade = len(produto.difference(formula))
  if quantidade == 0:
    return material
  return None

In [290]:
#If last code is true, this returns the Descrição name
def procura_produtos(formula, produtos, materiais):
  formula = formula.copy()
  resultados = []
  for prod, mat in zip(produtos, materiais):
    resultado = verifica_ingrediente(formula, prod, mat)
    if resultado is not None:
      formula = formula.difference(prod) ## Para remover os ingredientes já encontrados numa nova busca.
      resultados.append(resultado)
  return resultados

In [291]:
# Função para sinalizar ingredientes do dictOTHERS
def sinaliza_ingredientes(x):
    ingredientes = set().union(*x)  # une todos os sets/listas de ingredientes do grupo
    encontrados = set()
    for ing in ingredientes:
        for palavra in dictOTHERS:
            if palavra.lower() in ing.lower():
                encontrados.add(ing)
    return ', '.join(sorted(encontrados))

## Função Transpoe coluna

In [292]:
#this code transpose data. Used when we bring each category and the number of lauches.
def transpor (df, coluna, linha):
  for cat in df[coluna].unique():
    f = df[coluna] == cat
    df[cat] = df[f][linha]
    f = df[cat].isna()
    df.loc[f, cat] = df.loc[f, cat].apply(lambda x:[])

In [293]:
def to_set(x):
    s = set()
    for item in x:
        if isinstance(item, list):
            s.update(item)
        elif isinstance(item, str):
            s.add(item)
    return sorted(s)

In [294]:
def transpor_2 (df, coluna, linha):
  for cat in df[coluna].unique():
    f = df[coluna] == cat
    df[cat] = df[f][linha]
    f = df[cat].isna()
    df.loc[f, cat] = df.loc[f, cat].apply(lambda x:x)

## Função Estado


In [295]:
# Dicionário de siglas e nomes de estados
estados = {
    'AC': 'Acre', 'AL': 'Alagoas', 'AP': 'Amapá', 'AM': 'Amazonas', 'BA': 'Bahia', 'CE': 'Ceará',
    'DF': 'Distrito Federal', 'ES': 'Espírito Santo', 'GO': 'Goiás', 'MA': 'Maranhão', 'MT': 'Mato Grosso',
    'MS': 'Mato Grosso do Sul', 'MG': 'Minas Gerais', 'PA': 'Pará', 'PB': 'Paraíba', 'PR': 'Paraná',
    'PE': 'Pernambuco', 'PI': 'Piauí', 'RJ': 'Rio de Janeiro', 'RN': 'Rio Grande do Norte',
    'RS': 'Rio Grande do Sul', 'RO': 'Rondônia', 'RR': 'Roraima', 'SC': 'Santa Catarina',
    'SP': 'São Paulo', 'SE': 'Sergipe', 'TO': 'Tocantins'
}
# Inverte para buscar por nome também
estados_nome_para_sigla = {v.lower(): k for k, v in estados.items()}


In [296]:
def extrair_estado(texto):
    if pd.isnull(texto):
        return None
    texto = str(texto).strip().lower()
    # Procura por sigla
    for sigla in estados:
        if re.search(r'\b' + re.escape(sigla.lower()) + r'\b', texto):
            return sigla
    # Procura por nome do estado
    for nome, sigla in estados_nome_para_sigla.items():
        if nome in texto:
            return sigla
    return None

## Contém Silicone?

In [297]:
#Dicionário Silicones geral
dictOTHERS = {'methicone':'1', 'Dimethicone':'1','methicone Crosspolymer':'1','methiconol':'1',
              'ylsiloxysilicate':'1', 'ylsilsesquioxane':'1', 'Disiloxane':'1', 'Silica':'1','siloxane':'1'}

In [298]:
df_launches['Silicone'] = df_launches['Ingredients (Standard form)'].str.extract('('+'|'.join(dictOTHERS)+')',expand=False).map(dictOTHERS)

## Gera chave de Material


In [299]:
#Edita tabela INCI

df_inci.dropna(inplace=True)
df_inci['Produto'] = formatar_material(df_inci['Produto'])

#Cria uma lista iterável dos ingredientes nos Produtos
df_inci['Ing'] = df_inci['Ingrediente'].str.split(', ').apply(set)
df_inci.sort_values('Prioridade', ascending=True, inplace=True)

In [300]:
#Dicionário de palavras a remover da coluna Ingredientes no Mintel
dictIng = {
    r'\s*and/or\s*': ',',
    r',\s*': ',',
    r'\s*,': ',',
    r'\s*\(and\)\s*': ','
}

In [301]:
# Cria coluna com produtos identificados
df_launches['Ingredients (Standard form)'] = df_launches['Ingredients (Standard form)'].astype(str)
#Cria uma lista iterável das ingredientes cosmeticos
df_launches['Ing'] = (
    df_launches['Ingredients (Standard form)']
    .replace(dictIng, regex=True)
    .str.split(',')
    .apply(set)
)

# Relaciona os ingredientes cosméticos
df_launches['Prod Ident'] = df_launches['Ing'].apply(
    lambda formulacao: procura_produtos(formulacao, df_inci['Ing'], df_inci['Produto'])
)

## Gera Chave Fabricante



In [302]:
df_launches['Fabricante']= df_launches['Fabricante'].astype(str)
df_launches['Marca']= df_launches['Marca'].astype(str)
df_launches['Empresa']= df_launches['Empresa'].astype(str)
df_launches['Fabricante'] = df_launches['Fabricante'].fillna(df_launches['Empresa'])
df_launches['KeyManuf'] = gerar_chaves(df_launches, 'Fabricante', 'Empresa', 'Marca')
df_launches['KeyMarca'] = gerar_chaves(df_launches, 'Marca')
df_launches['KeyEmpresa'] = gerar_chaves(df_launches, 'Empresa')

Preencher os valores ausentes de 'Estado' com base em 'KeyManuf'

In [303]:
# Exemplo: supondo que sua coluna de empresa é 'Fabricante'
df_launches['Estado'] = df_launches['Manufacturer Company Address'].apply(extrair_estado)

In [304]:
# Preencher os valores ausentes de 'Estado' com base em 'KeyManuf'

missing_estado = df_launches[df_launches['Estado'].isna()]
for index, row in missing_estado.iterrows():
    keymanuf = row['KeyManuf']

    # Buscar registros com o mesmo 'KeyManuf' e 'Estado' não nulo
    estado_valido = df_launches[
        (df_launches['KeyManuf'] == keymanuf) &
        (df_launches['Estado'].notna())
    ]['Estado'].unique()

    # Se houver apenas um valor único de 'Estado', preencher
    if len(estado_valido) == 1:
        df_launches.at[index, 'Estado'] = estado_valido[0]

## Relatório de Lançamentos

In [305]:
df_launches.head()

,Número do Produto,Data de Publicação,Produto,Marca,Empresa,Categoria,Sub-Categoria,Descrição do Produto,Preço por 100g/ml,Preço em moeda local,Tipo de Lançamento,Posicionamento,Ingredients (Standard form),Fabricante,Território da empresa fabricante,Local de fabricação,Manufacturer Company Address,Link da Imagem Primária,Ano,Silicone,Ing,Prod Ident,KeyManuf,KeyMarca,KeyEmpresa,Estado
0,11306988,2024-01-02,Drying Gel,Needs Controle de Oleosidade,Instituto Pasteur de Cosmiatria,Produtos para Pele,Cuidado Facial/Pescoço,Needs Controle de Oleosidade Gel Secativo (Dry...,139.50,27.90,Nova Variedade/Extensão de Linha,"Botânico/Herbóreo, Antiacne, Testado Dermatolo...","Aqua, Alcohol, Glycerin, Propylene Glycol, Hyd...",Instituto Pasteur de Cosmiatria,Brazil,NaN,Porto Alegre - RS,https://media.mintel.com/i01/mediaserver/perfo...,2024,NaN,"{Propylene Glycol, Glycerin, Aloe Barbadensis ...",[],instituto,needs,instituto,RS
1,11369612,2024-01-02,Shampoo,Elefunte Baby Cachos Perfeitinhos,Cham's Industria de Cosméticos,Produtos para Cabelos,Xampu,Elefunte Baby Cachos Perfeitinhos (Perfect Cur...,6.79,13.58,Novo Produto,"Bebês e Crianças (0-4), Botânico/Herbóreo, Hip...","Aqua, Sodium Laureth Sulfate, Sodium Cocoampho...",Cham's Industria de Cosméticos,Brazil,Brasil,NaN,https://media.mintel.com/i01/mediaserver/perfo...,2024,NaN,"{Guar Hydroxypropyltrimonium Chloride, Coumari...",[],cham's,elefunte,cham's,None
2,11369614,2024-01-02,Conditioner,Elefunte Baby Cachos Perfeitinhos,Cham's Industria de Cosméticos,Produtos para Cabelos,Condicionador,Elefunte Baby Cachos Perfeitinhos Condicionado...,8.75,17.50,Novo Produto,"Bebês e Crianças (0-4), Fortificado com Vitami...","Aqua, Cetyl Alcohol, Stearamidopropyl Dimethyl...",Cham's Industria de Cosméticos,Brazil,Brasil,NaN,https://media.mintel.com/i01/mediaserver/perfo...,2024,1,"{Cetrimonium Chloride, Coumarin, Cinnamyl Alco...",[GB 1020],cham's,elefunte,cham's,None
3,11369618,2024-01-02,Conditioner,Elefunte Baby Cabelos Claros,Cham's Industria de Cosméticos,Produtos para Cabelos,Condicionador,Elefunte Baby Cabelos Claros Condicionador (Li...,8.75,17.50,Novo Produto,"Bebês e Crianças (0-4), Fortificado com Vitami...","Aqua, Cetyl Alcohol, Stearamidopropyl Dimethyl...",Cham's Industria de Cosméticos,Brazil,NaN,NaN,https://media.mintel.com/i01/mediaserver/perfo...,2024,1,"{Cetrimonium Chloride, Coumarin, Butylphenyl M...",[GB 1020],cham's,elefunte,cham's,None
4,11369620,2024-01-02,Shampoo,Elefunte Baby Hora de Dormir,Cham's Industria de Cosméticos,Produtos para Cabelos,Xampu,Elefunte Baby Hora de Dormir Shampoo is now av...,6.79,13.58,Novo Produto,"Bebês e Crianças (0-4), Botânico/Herbóreo, Hip...","Aqua, Sodium Laureth Sulfate, Sodium Cocoampho...",Cham's Industria de Cosméticos,Brazil,NaN,NaN,https://media.mintel.com/i01/mediaserver/perfo...,2024,NaN,"{Guar Hydroxypropyltrimonium Chloride, Coumari...",[],cham's,elefunte,cham's,None


In [306]:
df_launches['indice']=1
transpor_2(df_launches, 'Categoria', 'indice')

In [307]:
#renomeia coluna categoria
df_launches.rename(columns={'Produtos para Pele':'Pele', 'Produtos para Cabelos':'Cabelos',
                            'Maquilagem': 'Make',
                            'Número do Produto':'Total Lançamentos'},inplace=True)

In [308]:
df_c = df_launches.groupby(['KeyManuf', 'Estado'], dropna=False)[['Pele', 'Cabelos', 'Make']].apply(lambda x:x.count())

In [309]:
#cria tabela que lista o fabricante, o estado e o total de lançamentos, quais tem silicone,
df_b = df_launches.groupby(['KeyManuf', 'Estado'], dropna=False).agg({
    'KeyMarca': to_set,
    'KeyEmpresa': to_set,
    'Total Lançamentos': 'count',
    'Silicone': 'count',
    'Prod Ident': to_set,
    'Ing': sinaliza_ingredientes
})

In [310]:
#Une tabelas anteriores
df_launches_fab = pd.concat([df_c, df_b], axis=1).reset_index()

In [311]:
df_launches_fab.to_excel('Relatório Lançamentos.xlsx')
df_launches_fab.head()

,KeyManuf,Estado,Pele,Cabelos,Make,KeyMarca,KeyEmpresa,Total Lançamentos,Silicone,Prod Ident,Ing
0,&co,SP,17,0,5,"[alva, bars, beyoung, kind, ollie, pharmapele,...","[&co, bars, bscom, ollie]",23,17,"[ES 3007, GB 150, TMS 803]","C30-45 Alkyl Methicone, Cyclomethicone, Cyclop..."
1,a&a,PR,1,2,1,"[action, orobeauty, santo, vogue]",[a&a],5,3,[GB 1020],"Dimethicone, Dimethiconol, Silica, amodimethicone"
2,a'revalo,SP,0,3,0,[zap],[a'revalo],3,0,[],
3,abrahao,SP,2,0,1,[l'amazonie],"[cosmaidi, laboratoires]",3,3,"[ES 3007, GB 150, OW 2100]","Cyclopentasiloxane, Dimethicone, Dimethicone C..."
4,ac,SC,2,0,3,[oceane],"[ac, promex]",5,3,[ES 3007],"Cyclopentasiloxane, Dimethicone, Methicone, Si..."


## Agrupar relatório Vendas e Projetos

In [312]:
#agrupa lançamentos com vendas e projetos
df_launches_vend_proj = df_launches_fab.merge(df_final, on=['KeyManuf', 'Estado'], how='outer')

In [313]:
# Consolidar os dados por KeyManuf e Estado
df_consolidado = df_launches_vend_proj.groupby(['KeyManuf', 'Estado'], dropna=False).agg({
    'Pele': 'sum',
    'Cabelos': 'sum',
    'Make': 'sum',
    'Total Lançamentos': 'sum',
    'Silicone': 'sum',
    'KeyMarca': to_set,
    'KeyEmpresa':to_set,
    'Prod Ident':to_set,
    'Ing':to_set,
    'Distributor_Set': to_set,
    'Sales_Qnty_2025': 'sum',
    'Sales_Kg_2025': 'sum',
    'Sales_2025': to_set
})

In [314]:
df_consolidado.head()

,,Pele,Cabelos,Make,Total Lançamentos,Silicone,KeyMarca,KeyEmpresa,Prod Ident,Ing,Distributor_Set,Sales_Qnty_2025,Sales_Kg_2025
KeyManuf,Estado,,,,,,,,,,,,
&co,SP,17.0,0.0,5.0,23.0,17.0,"[alva, bars, beyoung, kind, ollie, pharmapele,...","[&co, bars, bscom, ollie]","[ES 3007, GB 150, TMS 803]","[C30-45 Alkyl Methicone, Cyclomethicone, Cyclo...",[FOCUS QUIMICA],3.0,398.0
a&a,PR,1.0,2.0,1.0,5.0,3.0,"[action, orobeauty, santo, vogue]",[a&a],[GB 1020],"[Dimethicone, Dimethiconol, Silica, amodimethi...",[],0.0,0.0
a&s,SP,0.0,0.0,0.0,0.0,0.0,[],[],[],[],[FOCUS QUIMICA],4.0,3814.0
a'revalo,SP,0.0,3.0,0.0,3.0,0.0,[zap],[a'revalo],[],[],[],0.0,0.0
abalo,SP,0.0,0.0,0.0,0.0,0.0,[],[],[],[],[FOCUS QUIMICA],6.0,60.0


# Generate a Report will all information

In [315]:
df_consolidado.to_excel('Final.xlsx')